# Análisis Comparativo de Técnicas de Mejora de Imagen

Este cuaderno documenta el análisis cuantitativo y cualitativo de los algoritmos de mejora de imagen implementados (CLAHE, Ecualización de Histograma y BHE2PL). El flujo de trabajo procesa las métricas extraídas (AMBE, PSNR, SSIM, Contraste y Entropía) para evaluar el rendimiento de cada técnica.

### Configuración Inicial e Importación de Bibliotecas

El siguiente bloque importa las bibliotecas necesarias para la manipulación matricial de los datos, la visualización científica y el análisis estadístico posterior.

In [48]:
import json
import pandas as pd

# Bibliotecas para visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Biblioteca para análisis estadístico riguroso
import scipy.stats as stats

# Configuración global del estilo para gráficos
sns.set_theme(style="whitegrid")

### Carga y Transformación de Datos

En esta sección se lee el registro estructurado de los experimentos desde el disco. El código utiliza la función de aplanado de Pandas para transformar los diccionarios anidados del JSON en un DataFrame bidimensional. Esta vectorización prepara las métricas (PSNR, AMBE, SSIM, Contraste y Entropía) para el análisis estadístico posterior, eliminando la necesidad de bucles iterativos ineficientes.

In [49]:
# Definición de la ruta relativa al archivo JSON generado por el orquestador
ruta_archivo_json = '../experimento/json/resultados_evaluacion.json'

# Lectura del archivo en crudo utilizando el manejador de contexto
with open(ruta_archivo_json, 'r', encoding='utf-8') as archivo:
    datos_crudos = json.load(archivo)

# Transformación vectorizada: aplanado de diccionarios anidados a formato tabular
marco_datos = pd.json_normalize(datos_crudos)

In [50]:
# Visualización de las primeras 5 filas
marco_datos.head()

,id_imagen,metodo,rutas.original,rutas.oscurecida,rutas.procesada,metricas_imagen_procesada.PSNR,metricas_imagen_procesada.AMBE,metricas_imagen_procesada.SSIM,metricas_imagen_procesada.Contraste,metricas_imagen_procesada.Entropia,metricas_imagen_procesada.tiempo_ms,metricas_imagen_oscurecida.PSNR,metricas_imagen_oscurecida.AMBE,metricas_imagen_oscurecida.SSIM,metricas_imagen_oscurecida.Contraste,metricas_imagen_oscurecida.Entropia,metricas_imagen_original.Contraste,metricas_imagen_original.Entropia,parametros.limite_contraste,parametros.tamano_cuadricula
0,0001_HE_Global.png,HE_Global,dataset/img/original/0001.png,dataset/img/oscurecida/0001.png,experimento/img/0001_HE_Global.png,11.958643,55.644603,0.351285,70.614801,3.324304,89.3153,9.160244,76.461957,0.050448,3.071889,3.32889,48.085538,7.253181,NaN,NaN
1,0001_CLAHE_2_8.png,CLAHE_2_8,dataset/img/original/0001.png,dataset/img/oscurecida/0001.png,experimento/img/0001_CLAHE_2_8.png,10.225435,67.298417,0.222998,8.173766,4.336241,15.6845,9.160244,76.461957,0.050448,3.071889,3.32889,48.085538,7.253181,2.0,"[8, 8]"
2,0001_CLAHE_2_4.png,CLAHE_2_4,dataset/img/original/0001.png,dataset/img/oscurecida/0001.png,experimento/img/0001_CLAHE_2_4.png,10.294284,66.817841,0.228264,8.560625,4.127718,0.9051,9.160244,76.461957,0.050448,3.071889,3.32889,48.085538,7.253181,2.0,"[4, 4]"
3,0001_CLAHE_4_8.png,CLAHE_4_8,dataset/img/original/0001.png,dataset/img/oscurecida/0001.png,experimento/img/0001_CLAHE_4_8.png,11.267481,59.540009,0.349484,13.083222,4.961323,0.8208,9.160244,76.461957,0.050448,3.071889,3.32889,48.085538,7.253181,4.0,"[8, 8]"
4,0001_BHE2PL.png,BHE2PL,dataset/img/original/0001.png,dataset/img/oscurecida/0001.png,experimento/img/0001_BHE2PL.png,9.181666,76.261030,0.052421,3.149257,3.169954,35.0934,9.160244,76.461957,0.050448,3.071889,3.32889,48.085538,7.253181,NaN,NaN


In [51]:
# Obtenemos los nombres de las columnas
marco_datos.columns

Index(['id_imagen', 'metodo', 'rutas.original', 'rutas.oscurecida',
       'rutas.procesada', 'metricas_imagen_procesada.PSNR',
       'metricas_imagen_procesada.AMBE', 'metricas_imagen_procesada.SSIM',
       'metricas_imagen_procesada.Contraste',
       'metricas_imagen_procesada.Entropia',
       'metricas_imagen_procesada.tiempo_ms',
       'metricas_imagen_oscurecida.PSNR', 'metricas_imagen_oscurecida.AMBE',
       'metricas_imagen_oscurecida.SSIM',
       'metricas_imagen_oscurecida.Contraste',
       'metricas_imagen_oscurecida.Entropia',
       'metricas_imagen_original.Contraste',
       'metricas_imagen_original.Entropia', 'parametros.limite_contraste',
       'parametros.tamano_cuadricula'],
      dtype='str')

In [52]:
# Diccionario de mapeo para aislar y renombrar las métricas

columnas_renombradas = {
    # Identificadores
    'id_imagen': 'id_imagen',
    'metodo': 'metodo',
    # Métricas de la imagen procesada
    'metricas_imagen_procesada.PSNR': 'psnr',
    'metricas_imagen_procesada.AMBE': 'ambe',
    'metricas_imagen_procesada.SSIM': 'ssim',
    'metricas_imagen_procesada.Contraste': 'contraste',
    'metricas_imagen_procesada.Entropia': 'entropia',
    'metricas_imagen_procesada.tiempo_ms': 'tiempo_ms',
    # Métricas de la imagen oscurecida
    'metricas_imagen_oscurecida.PSNR': 'psnr_base',
    'metricas_imagen_oscurecida.AMBE': 'ambe_base',
    'metricas_imagen_oscurecida.SSIM': 'ssim_base',
    'metricas_imagen_oscurecida.Contraste': 'contraste_base',
    'metricas_imagen_oscurecida.Entropia': 'entropia_base',
    # Métricas de la imagen original
    'metricas_imagen_original.Contraste': 'contraste_orig',
    'metricas_imagen_original.Entropia': 'entropia_orig',
    # Parámetros adicionales (puede tener valores NaN)
    'parametros.limite_contraste': 'limite_contraste',
    'parametros.tamano_cuadricula': 'tamano_cuadricula'
}

# Aplicación del filtro y renombrado en una sola operación de Pandas
marco_datos_limpio = marco_datos.rename(columns=columnas_renombradas)[columnas_renombradas.values()]

In [53]:
# Obtenemos los nombres de las columnas
marco_datos_limpio.columns

Index(['id_imagen', 'metodo', 'psnr', 'ambe', 'ssim', 'contraste', 'entropia',
       'tiempo_ms', 'psnr_base', 'ambe_base', 'ssim_base', 'contraste_base',
       'entropia_base', 'contraste_orig', 'entropia_orig', 'limite_contraste',
       'tamano_cuadricula'],
      dtype='str')

In [54]:
# Visualización de las primeras 5 filas
marco_datos_limpio.head()

,id_imagen,metodo,psnr,ambe,ssim,contraste,entropia,tiempo_ms,psnr_base,ambe_base,ssim_base,contraste_base,entropia_base,contraste_orig,entropia_orig,limite_contraste,tamano_cuadricula
0,0001_HE_Global.png,HE_Global,11.958643,55.644603,0.351285,70.614801,3.324304,89.3153,9.160244,76.461957,0.050448,3.071889,3.32889,48.085538,7.253181,NaN,NaN
1,0001_CLAHE_2_8.png,CLAHE_2_8,10.225435,67.298417,0.222998,8.173766,4.336241,15.6845,9.160244,76.461957,0.050448,3.071889,3.32889,48.085538,7.253181,2.0,"[8, 8]"
2,0001_CLAHE_2_4.png,CLAHE_2_4,10.294284,66.817841,0.228264,8.560625,4.127718,0.9051,9.160244,76.461957,0.050448,3.071889,3.32889,48.085538,7.253181,2.0,"[4, 4]"
3,0001_CLAHE_4_8.png,CLAHE_4_8,11.267481,59.540009,0.349484,13.083222,4.961323,0.8208,9.160244,76.461957,0.050448,3.071889,3.32889,48.085538,7.253181,4.0,"[8, 8]"
4,0001_BHE2PL.png,BHE2PL,9.181666,76.261030,0.052421,3.149257,3.169954,35.0934,9.160244,76.461957,0.050448,3.071889,3.32889,48.085538,7.253181,NaN,NaN


In [55]:
# Inspección de estructuras de datos y verificación de tipos numéricos
marco_datos_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 4250 entries, 0 to 4249
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_imagen          4250 non-null   str    
 1   metodo             4250 non-null   str    
 2   psnr               4250 non-null   float64
 3   ambe               4250 non-null   float64
 4   ssim               4250 non-null   float64
 5   contraste          4250 non-null   float64
 6   entropia           4250 non-null   float64
 7   tiempo_ms          4250 non-null   float64
 8   psnr_base          4250 non-null   float64
 9   ambe_base          4250 non-null   float64
 10  ssim_base          4250 non-null   float64
 11  contraste_base     4250 non-null   float64
 12  entropia_base      4250 non-null   float64
 13  contraste_orig     4250 non-null   float64
 14  entropia_orig      4250 non-null   float64
 15  limite_contraste   2550 non-null   float64
 16  tamano_cuadricula  2550 non-null   

### Estadística Descriptiva por Método de Mejora

En esta sección se agrupa las métricas cuantitativas por algoritmo (CLAHE, Ecualización de Histograma y BHE2PL) utilizando operaciones vectorizadas. El análisis genera dos matrices fundamentales para la evaluación del rendimiento. La Tabla 1 presenta las medias para evaluar la centralidad de cada métrica sobre el conjunto de datos. La Tabla 2 expone la desviación estándar para analizar la dispersión y estabilidad de cada técnica.

In [56]:
def generar_estadistica_descriptiva(marco_datos, columnas):
    """
    Agrupa los datos por método de mejora de imagen y calcula las medidas de tendencia 
    central y dispersión.
    
    Args:
        marco_datos: Marco de datos tabular con los resultados extraídos del procesamiento.
        columnas: Lista de cadenas con los nombres de las métricas a evaluar.
        
    Returns:
        Una tupla que contiene la tabla de medias transpuesta y la tabla de desviaciones 
        estándar transpuesta.
    """
    # Agrupación vectorizada
    datos_agrupados = marco_datos.groupby('metodo')[columnas]
    
    # Cálculo, transposición y redondeo en operaciones encadenadas eficientes
    tabla_medias = datos_agrupados.mean().T.round(4)
    tabla_desviaciones = datos_agrupados.std().T.round(4)
    
    return tabla_medias, tabla_desviaciones

A continuación, ejecutamos la función de estadística descriptiva sobre las métricas evaluadas (PSNR, AMBE, Contraste, Entropía, SSIM y Tiempo). La Tabla 1 muestra las medias obtenidas para cada algoritmo. La Tabla 2 presenta las desviaciones estándar correspondientes.

In [57]:
# Definición de las columnas numéricas exigidas (AMBE, PSNR, Contraste, Entropía) 
# y adicionales para puntos extra
columnas_metricas = ['psnr', 'ambe', 'contraste', 'entropia', 'ssim', 'tiempo_ms']

# Ejecución de la función encapsulada
tabla_medias, tabla_desviaciones = generar_estadistica_descriptiva(marco_datos_limpio, columnas_metricas)

In [58]:
# Visualización de la tabla 1
print("\nTabla 1: Medias de Métricas por Algoritmo\n")
tabla_medias


Tabla 1: Medias de Métricas por Algoritmo



metodo,BHE2PL,CLAHE_2_4,CLAHE_2_8,CLAHE_4_8,HE_Global
psnr,8.7150,10.4698,10.2043,11.5168,13.3562
ambe,85.7802,71.1661,73.0221,63.1641,41.8876
contraste,8.1599,16.7921,15.7184,21.5759,72.1710
entropia,3.8626,5.1001,5.0756,5.6612,3.9127
ssim,0.1320,0.3382,0.3198,0.4266,0.3968
tiempo_ms,0.7281,0.9142,0.8993,0.8672,0.5587


La Tabla 1 presenta las medias de las métricas de calidad de imagen. Los parámetros de configuración son fundamentales para la reproducibilidad de estos experimentos. La nomenclatura `CLAHE_4_8` indica un límite de contraste igual a 4.0 y una división de cuadrícula de 8x8 píxeles. La nomenclatura `CLAHE_2_4` representa un límite de contraste de 2.0 y una cuadrícula de 4x4 píxeles.

El algoritmo de Ecualización de Histograma Global (HE_Global) obtuvo el menor Error Absoluto Medio de Brillo (AMBE = 41.8876). Este método registró también el valor más alto en la Relación Señal a Ruido Máxima (PSNR = 13.3562). Estos resultados indican una menor distorsión global promedio respecto a la imagen original. En contraste, el algoritmo BHE2PL presentó el mayor error de preservación de brillo (AMBE = 85.7802).

El método HE_Global maximizó el contraste de las imágenes (Contraste = 72.1710). Sin embargo, este algoritmo redujo fuertemente la cantidad de información visual (Entropía = 3.9127). El algoritmo CLAHE con límite de contraste 4.0 y cuadrícula de 8x8 (CLAHE_4_8) obtuvo la mayor entropía de la evaluación (Entropía = 5.6612). Este resultado demuestra numéricamente que la técnica CLAHE preserva los detalles locales de manera superior a las técnicas globales. 

El método CLAHE_4_8 alcanzó el mejor Índice de Similitud Estructural (SSIM = 0.4266). Este valor confirma su capacidad superior para mantener la estructura visual original. Finalmente, HE_Global registró el menor tiempo de procesamiento (0.5587 ms). Las distintas configuraciones de CLAHE requirieron un mayor tiempo computacional. Este retraso ocurre debido a los cálculos de interpolación bilineal efectuados entre las particiones de la cuadrícula.

Con base en los datos extraídos, se recomienda el uso de HE_Global para aplicaciones rápidas que requieren un contraste extremo. Se recomienda implementar la técnica CLAHE_4_8 para tareas analíticas que exigen una retención estricta de los detalles locales y estructurales.

In [59]:
# Visualización de la tabla 2
print("\nTabla 2: Desviación Estándar de Métricas por Algoritmo\n")
tabla_desviaciones


Tabla 2: Desviación Estándar de Métricas por Algoritmo



metodo,BHE2PL,CLAHE_2_4,CLAHE_2_8,CLAHE_4_8,HE_Global
psnr,1.8912,2.4471,2.3598,2.9016,3.0713
ambe,18.6731,19.4628,19.4001,20.4426,21.4511
contraste,6.1581,9.4993,9.0922,10.6878,4.5940
entropia,0.8592,0.9338,0.8808,0.8668,0.8278
ssim,0.0850,0.1463,0.1391,0.1503,0.1507
tiempo_ms,1.1883,0.4379,0.5731,0.2725,3.0482


La Tabla 2 presenta las desviaciones estándar de las métricas evaluadas. Un valor estadístico menor indica una mayor estabilidad del algoritmo sobre el conjunto de datos. 

El algoritmo BHE2PL presenta la menor desviación estándar en el Error Absoluto Medio de Brillo (AMBE = 18.6731). Este método registra también la menor dispersión en la Relación Señal a Ruido Máxima (PSNR = 1.8912). Estos datos revelan un comportamiento altamente consistente del algoritmo BHE2PL frente a distintas imágenes de entrada. La Ecualización de Histograma Global (HE_Global) exhibe la mayor inestabilidad en la preservación del brillo (AMBE = 21.4511).

El método HE_Global genera el aumento de contraste más uniforme del estudio (Contraste = 4.5940). El algoritmo CLAHE_4_8 muestra la mayor variabilidad en la mejora del contraste (Contraste = 10.6878). El método HE_Global proporciona la mayor estabilidad en la retención de la información visual (Entropía = 0.8278).

El algoritmo BHE2PL es el método más estable para mantener la estructura visual original (SSIM = 0.0850). El método CLAHE_4_8 ofrece el tiempo de procesamiento computacional más predecible (Tiempo = 0.2725 ms). El algoritmo HE_Global fluctúa drásticamente en su rendimiento temporal (Tiempo = 3.0482 ms).

El estudio concluye que BHE2PL proporciona la mayor robustez general frente a diferentes imágenes. Las variaciones de la técnica CLAHE presentan fluctuaciones métricas moderadas. Esta variabilidad de CLAHE responde correctamente a su naturaleza matemática de adaptación local de histogramas.

### Análisis de Mejora Relativa frente a la Línea Base (Deltas)

En esta sección se evalúa el impacto cuantitativo de los algoritmos de mejora sobre las imágenes degradadas. Para lograrlo, se calcula la diferencia matemática (delta) entre las métricas de la imagen procesada y las métricas de la línea base  (imagen oscurecida). 

El cálculo del delta para el Índice de Similitud Estructural (SSIM) y la Relación Señal a Ruido Máxima (PSNR) cuantifica la capacidad de restauración de cada método. La diferencia en las métricas de contraste y entropía mide el volumen de información visual inyectada por el algoritmo. Finalmente, la reducción del Error Absoluto Medio de Brillo (AMBE) indica la disminución de la distorsión respecto a la captura original.

In [60]:
def calcular_mejora_relativa(marco_datos):
    """
    Calcula la diferencia matemática (delta) entre las métricas procesadas 
    y las métricas de la línea base (imagen oscurecida).
    
    Args:
        marco_datos: Marco de datos tabular con los resultados extraídos del experimento.
        
    Returns:
        Un nuevo marco de datos que incluye las columnas calculadas de mejora relativa.
    """
    # Generación de una copia para preservar la inmutabilidad del conjunto de datos original
    datos_analisis = marco_datos.copy()
    
    datos_analisis['delta_psnr'] = datos_analisis['psnr'] - datos_analisis['psnr_base']
    datos_analisis['delta_ssim'] = datos_analisis['ssim'] - datos_analisis['ssim_base']
    datos_analisis['delta_contraste'] = datos_analisis['contraste'] - datos_analisis['contraste_base']
    datos_analisis['delta_entropia'] = datos_analisis['entropia'] - datos_analisis['entropia_base']

    # Para el AMBE, una mejora técnica implica la reducción matemática del error de brillo
    datos_analisis['reduccion_ambe'] = datos_analisis['ambe_base'] - datos_analisis['ambe']
    
    return datos_analisis

El siguiente bloque de código aplica la función de cálculo sobre el conjunto de datos limpio y define las columnas de interés para el análisis agrupado.

In [61]:
# Ejecución de la función sobre el marco de datos limpio
marco_datos_deltas = calcular_mejora_relativa(marco_datos_limpio)

# Definición de las nuevas métricas relativas para la agrupación
columnas_deltas = [
    'delta_psnr', 
    'reduccion_ambe', 
    'delta_contraste', 
    'delta_entropia', 
    'delta_ssim'
]

La Tabla 3 presenta las medias de las mejoras relativas (deltas) obtenidas para cada técnica. Un valor positivo en la reducción del AMBE indica una mejora en la preservación del brillo original, mientras que los incrementos en PSNR y SSIM demuestran una recuperación estructural superior frente a la imagen subexpuesta.

In [62]:
# Cálculo vectorizado de las medias agrupadas por algoritmo
tabla_medias_deltas = marco_datos_deltas.groupby('metodo')[columnas_deltas].mean().T.round(4)

# Visualización del resultado
print("\nTabla 3: Medias de Mejoras Relativas (Deltas) por Algoritmo\n")
tabla_medias_deltas


Tabla 3: Medias de Mejoras Relativas (Deltas) por Algoritmo



metodo,BHE2PL,CLAHE_2_4,CLAHE_2_8,CLAHE_4_8,HE_Global
delta_psnr,0.0045,1.7593,1.4939,2.8063,4.6457
reduccion_ambe,0.0235,14.6376,12.7816,22.6396,43.9161
delta_contraste,0.0847,8.7168,7.6432,13.5007,64.0958
delta_entropia,-0.0839,1.1536,1.1291,1.7147,-0.0338
delta_ssim,0.0004,0.2066,0.1881,0.2950,0.2651


La Tabla 3 expone el impacto cuantitativo de cada algoritmo frente a la imagen original oscurecida. El método HE_Global logró la mayor reducción del Error Absoluto Medio de Brillo (reducción de 43.9161) y el mayor incremento en la Relación Señal a Ruido Máxima (aumento de 4.6457). Este algoritmo inyectó una cantidad extrema de contraste (aumento de 64.0958). Sin embargo, la técnica HE_Global produjo una pérdida de información visual, evidenciada por un delta de entropía negativo (-0.0338).

Las variantes del algoritmo CLAHE demostraron un comportamiento más equilibrado. La configuración CLAHE_4_8 (límite de contraste 4.0 y cuadrícula de 8x8) presentó el mayor incremento en el Índice de Similitud Estructural (aumento de 0.2950) y la mayor ganancia de información (aumento de entropía de 1.7147). Estos valores numéricos confirman que la técnica CLAHE restaura los detalles locales de manera efectiva sin saturar globalmente la imagen. 

El algoritmo BHE2PL registró deltas cercanos a cero en todas las métricas evaluadas. Este resultado cuantitativo indica que el método BHE2PL no logró mejorar significativamente la imagen degradada respecto a su estado base. 

En conclusión, el comportamiento de estas variaciones justifica recomendaciones de uso distintas. Se recomienda la aplicación de CLAHE para tareas que requieren una preservación estricta de detalles estructurales. Se recomienda el uso de HE_Global únicamente para escenarios que exigen maximizar el contraste puro sin importar la pérdida de microtexturas.

### Análisis de Recuperación Informativa (Contraste y Entropía)

En esta sección se analiza la evolución del contraste y la entropía a través de los tres estados experimentales: la imagen original, la imagen oscurecida (línea base) y la imagen procesada. Este análisis bidireccional permite evaluar la capacidad real de los algoritmos para restaurar la riqueza informativa de la escena sin introducir ruido ni artefactos artificiales.

El siguiente bloque implementa la función que calcula las tasas relativas de recuperación de contraste y entropía.

In [63]:
def calcular_tasas_recuperacion(marco_datos):
    """
    Calcula las tasas relativas de recuperación de contraste y entropía 
    comparando el estado procesado frente a la línea base y la imagen original.
    
    Args:
        marco_datos: Marco de datos tabular con los resultados del experimento.
        
    Returns:
        Un nuevo marco de datos con las métricas de recuperación añadidas.
    """
    datos_analisis = marco_datos.copy()
    
    # Riqueza informativa retenida o amplificada frente a la original
    datos_analisis['ratio_entropia'] = datos_analisis['entropia'] / datos_analisis['entropia_orig']
    
    # Magnitud de expansión del contraste respecto al daño inicial sufrida por la base
    datos_analisis['ganancia_contraste'] = datos_analisis['contraste'] - datos_analisis['contraste_base']
    
    return datos_analisis

A continuación, se ejecuta la función sobre el conjunto de datos depurado y se seleccionan tanto las métricas absolutas como las derivadas para su estructuración tabular.

In [66]:
# Ejecución sobre el conjunto de datos limpio
marco_datos_recuperacion = calcular_tasas_recuperacion(marco_datos_limpio)

# Selección de columnas para la tabla resumen de recuperación
columnas_recuperacion = ['contraste', 'contraste_base', 'entropia','entropia_orig', 'ganancia_contraste', 'ratio_entropia']

La Tabla 4 resume las medias absolutas y relativas obtenidas por cada técnica de mejora, permitiendo contrastar el nivel de recuperación frente a los estados original y degradado.

In [67]:
# Agrupación vectorizada de las medias por algoritmo y transposición de la matriz
tabla_medias_recuperacion = marco_datos_recuperacion.groupby('metodo')[columnas_recuperacion].mean().T.round(4)

# Visualización de la matriz resultante
print("\nTabla 4: Medias de Contraste, Entropía y Tasas de Recuperación por Algoritmo\n")
display(tabla_medias_recuperacion)


Tabla 4: Medias de Contraste, Entropía y Tasas de Recuperación por Algoritmo



metodo,BHE2PL,CLAHE_2_4,CLAHE_2_8,CLAHE_4_8,HE_Global
contraste,8.1599,16.7921,15.7184,21.5759,72.1710
contraste_base,8.0752,8.0752,8.0752,8.0752,8.0752
entropia,3.8626,5.1001,5.0756,5.6612,3.9127
entropia_orig,6.8529,6.8529,6.8529,6.8529,6.8529
ganancia_contraste,0.0847,8.7168,7.6432,13.5007,64.0958
ratio_entropia,0.5625,0.7458,0.7421,0.8287,0.5698


La Tabla 4 permite analizar cómo cada técnica afecta la calidad y la percepción visual de las imágenes. El método HE_Global generó la mayor ganancia de contraste absoluto. Esta técnica alcanzó un incremento de 64.0958 respecto a la línea base. Sin embargo, el algoritmo registró un bajo ratio de retención de entropía (0.5698). Este comportamiento confirma que la ecualización global expande agresivamente los niveles de intensidad. Este proceso destructivo elimina gran parte de las microestructuras de la escena original y produce imágenes artificialmente saturadas.

Las variaciones del algoritmo CLAHE mostraron una retención muy superior de la información. El método CLAHE_4_8 obtuvo el mayor ratio de entropía de todo el experimento. Esta técnica recuperó exitosamente el 82.87% de la riqueza visual original. Además, este algoritmo inyectó una ganancia de contraste moderada e ideal de 13.5007. Estos valores justifican fuertemente la recomendación de usar CLAHE para realzar imágenes sin provocar una sobresaturación.

El algoritmo BHE2PL presentó el desempeño más deficiente del análisis. Este método logró una ganancia de contraste casi nula (0.0847). La técnica retuvo únicamente el 56.25% de la entropía original. Este resultado evidencia una falta de capacidad para procesar imágenes con este grado de subexposición. 

En conclusión, la discusión de estos resultados recomienda implementar la configuración CLAHE_4_8. Este algoritmo ofrece el equilibrio matemático perfecto entre la inyección de contraste y la preservación estricta de los detalles estructurales.